# 05 Baseline B0 on the development pool (Prompt 09, cycles 5–6)

**Needs a GPU**: Runtime → Change runtime type → **T4 GPU**. First upload into the Drive project folder: `src/evaluate.py`, `src/plots.py`, `scripts/evaluate_condition.py`, `tests/test_evaluate.py`, `notebooks/05_dev_baseline.ipynb`.

Then Runtime → **Run all**. Cell 3 runs 500 generations (about 40 minutes). **Keep this tab open.** If Colab disconnects, reconnect and Run all again: finished answers are skipped and it continues where it stopped. Only **development** items are used. Copy all the output (and a screenshot of the two figures) back into the chat.

In [ ]:
# 1. Drive, project folder, libraries
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/atr_colab/AITR_Lab02_MCL2026002_PromptStability
!pip install -q "transformers>=4.37.0" accelerate bitsandbytes pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Tests for the evaluation module (no GPU needed)
!python tests/test_evaluate.py

In [ ]:
# 3. B0 on all 500 development items: P0, greedy (resumable)
!python scripts/run_generation.py --condition dev_b0 --prompts P0 2>&1 | grep -v -E 'Downloading|Reconstructing|Fetching|Download complete|Reconstruction complete'

In [ ]:
# 4. Evaluate, save predictions, tables and figures
!python scripts/evaluate_condition.py --condition dev_b0
!ls results/predictions results/tables results/figures

In [ ]:
# 5. Show the two figures
from IPython.display import Image, display
display(Image('results/figures/dev_b0_confusion.png', width=520))
display(Image('results/figures/dev_b0_labels.png', width=520))

In [ ]:
# 6. Any invalid outputs, shown in full (expected: few or none)
import json
m = json.load(open('results/tables/dev_b0_metrics.json'))
bad = set(m['invalid_run_ids'])
print(len(bad), 'invalid')
for line in open('results/generations/dev_b0.jsonl'):
    r = json.loads(line)
    if r['run_id'] in bad:
        print('-----', r['run_id']); print(r['raw_text'])